# 01 · Exploring the data

This notebook looks at the data before any machine learning. By the end you should understand:

1. **The labels**: what a TCE is, how many of each class there are, and how the classes differ in simple numbers.
2. **The light curves**: what Kepler actually measured, and whether your downloads are healthy.
3. **The transformation** from a raw light curve into the two pictures the CNN will see: a *global view* (the whole orbit) and a *local view* (zoomed on the transit).
4. **What separates planets from false positives** by eye. This is what the network has to learn.

Run the cells from top to bottom. Each section ends with a short **"What you should see"** note.

## 0 · Setup

In [ ]:
import sys, os
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "download.py").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binned_statistic

import lightkurve as lk
from astropy.time import Time

from src.download import load_labelled, SUBSET_CSV, LC_DIR, lightcurve_is_bad

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.figsize": (11, 4), "axes.grid": True, "grid.alpha": 0.3})
COLORS = {"PC": "tab:green", "AFP": "tab:red", "NTP": "tab:gray"}
print("Project folder:", ROOT)

## 1 · The labels

Each row is one **TCE (Threshold Crossing Event)**: a periodic dip that Kepler's software found in one star's light curve.
Humans then vetted about 15,700 of them into three classes:

| Label | Meaning | In our model |
|---|---|---|
| **PC** | planet candidate | 1 (planet) |
| **AFP** | astrophysical false positive, mostly eclipsing binary stars | 0 |
| **NTP** | non-transiting phenomenon: noise, instrument artefacts, star spots | 0 |

Useful columns: `tce_period` (days), `tce_duration` (**hours**), `tce_depth` (ppm, parts per million: 1,000 ppm = 0.1% dimmer), `tce_model_snr` (signal-to-noise), `tce_impact` (0 = transit crosses the centre of the star, ~1 = grazes the edge).

In [ ]:
df = load_labelled()
print(f"{len(df):,} labelled TCEs on {df.kepid.nunique():,} stars")
print(f"planet share: {df.label.mean():.1%}\n")
print(df.av_training_set.value_counts().to_string())
df.head()

In [ ]:
per_star = df.groupby("kepid").size()
print("TCEs per star:")
print(per_star.value_counts().sort_index().to_string())
print(f"\n{(per_star > 1).mean():.1%} of stars have more than one TCE")

**What you should see:** roughly 23% planets, and a noticeable share of stars with 2 or more TCEs.

That second number is why the train/test split must be **by star**. Several TCEs on one star share the same noise and the same light curve. If one lands in training and another in testing, the model is partly tested on something it has already seen.

## 2 · How the classes differ in simple numbers

Before looking at any light curves: can the summary numbers alone separate the classes? Everything is on a log scale, because periods range from under a day to hundreds of days and depths over five orders of magnitude.

In [ ]:
cols = [("tce_period", "period [days]"), ("tce_depth", "depth [ppm]"),
        ("tce_duration", "duration [hours]"), ("tce_model_snr", "signal-to-noise")]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (col, name) in zip(axes, cols):
    x = df[col].clip(lower=1e-2)
    bins = np.logspace(np.log10(x.quantile(0.005)), np.log10(x.quantile(0.995)), 50)
    for lab in ["PC", "AFP", "NTP"]:
        ax.hist(x[df.av_training_set == lab], bins=bins, histtype="step", lw=1.8, color=COLORS[lab], label=lab, density=True)
    ax.set_xscale("log"); ax.set_xlabel(name); ax.set_yticks([])
axes[0].legend(); plt.tight_layout()

**What you should see:**

- **Depth:** planets (green) are mostly **shallow**, roughly 100 to 10,000 ppm (0.01% to 1%). A planet is much smaller than its star. Eclipsing binaries (red) reach much **deeper**, often more than 10,000 ppm, because a whole star blocks light. So depth alone already says a lot.
- **Signal-to-noise:** non-transiting phenomena (grey) pile up at **low SNR**. Many of them are noise that only just crossed the detection threshold.
- **Period:** the classes overlap a lot. Period alone separates little.

The overlap is the point. Plenty of shallow AFPs (a binary blended with a brighter star) and deep PCs (a large planet around a small star) exist. That is why we need the **shape** of the dip, not just its size.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
for lab in ["NTP", "AFP", "PC"]:
    d = df[df.av_training_set == lab]
    ax.scatter(d.tce_period, d.tce_depth.clip(lower=1), s=3, alpha=0.35, color=COLORS[lab], label=lab)
ax.set(xscale="log", yscale="log", xlabel="period [days]", ylabel="depth [ppm]", title="Every labelled TCE")
ax.legend(markerscale=5); plt.tight_layout()

**What you should see:** green planets in a band at the bottom (shallow), red binaries spread upwards, grey noise scattered with the green.

A model on these numbers alone is **baseline 1** in the plan. The next cell trains one in seconds, so you know the score the CNN has to beat.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

features = ["tce_period", "tce_depth", "tce_duration", "tce_model_snr", "tce_impact"]
X = np.log10(df[features].clip(lower=1e-3))
y = df.label

train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0).split(X, y, groups=df.kepid))
clf = HistGradientBoostingClassifier(random_state=0).fit(X.iloc[train_idx], y.iloc[train_idx])
auc = roc_auc_score(y.iloc[test_idx], clf.predict_proba(X.iloc[test_idx])[:, 1])
print(f"Baseline (5 numbers, no light curve): test AUC = {auc:.3f}")

**What you should see:** an AUC around 0.96. That's high for five numbers, mostly because depth alone gives so much away. The CNN's target is about 0.98, so the light curve has to win the last few points: the hard cases where the numbers look the same. Keep this number: it's the bar the light-curve models have to clear.

*(This is a quick check with one random split. The real baseline goes in `src/` with the saved split, so the comparison is fair.)*

## 3 · Are the downloads healthy?

Each star's light curve is saved as `data/raw/lightcurves/<kepid>.npz` with three arrays: `time` (days, BKJD), `flux` (brightness, normalised so the typical value is 1) and `flux_err`.

In [ ]:
sub = pd.read_csv(SUBSET_CSV)
files = {int(p.stem): p for p in LC_DIR.glob("*.npz")}
sub["have_lc"] = sub.kepid.isin(files)
print(f"subset: {len(sub)} TCEs on {sub.kepid.nunique()} stars")
print(f"light curves on disk: {len(files)}  ->  {sub.have_lc.mean():.1%} of subset TCEs covered")

stats = []
for k, p in files.items():
    d = np.load(p)
    stats.append({"kepid": k, "n_points": len(d["time"]), "span_days": d["time"].max() - d["time"].min(),
                  "scatter_ppm": 1e6 * np.nanstd(np.diff(d["flux"])) / np.sqrt(2), "bad": lightcurve_is_bad(p)})
stats = pd.DataFrame(stats)
print(f"files flagged as broken: {stats.bad.sum()}")

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
ax[0].hist(stats.n_points, bins=50); ax[0].set_xlabel("points per light curve")
ax[1].hist(stats.span_days, bins=50); ax[1].set_xlabel("time span [days]")
ax[2].hist(np.log10(stats.scatter_ppm), bins=50); ax[2].set_xlabel("log10 point-to-point scatter [ppm]")
plt.tight_layout()

**What you should see:**

- **Points per light curve:** a big peak around 60,000 to 65,000. That's about 4 years at one point every 29.4 minutes, minus gaps. A tail of smaller values comes from stars observed for only part of the mission, or with broken quarters removed.
- **Time span:** a peak near 1,460 days (the full mission).
- **Scatter:** mostly 30 to 1,000 ppm. Bright stars are quieter than faint ones. A planet with a 100 ppm transit is invisible in a single transit around a 1,000 ppm star. Folding many transits together (section 5) is what makes it visible.
- **Broken files:** should be **0** if you ran `python -m src.download check --delete`.

## 4 · One light curve, step by step

Pick one planet candidate and follow it through the whole pipeline. These helper functions are an early version of what goes into `src/preprocess.py`.

In [ ]:
def load_lc(kepid):
    d = np.load(LC_DIR / f"{kepid}.npz")
    return lk.LightCurve(time=Time(d["time"], format="bkjd", scale="tdb"), flux=d["flux"], flux_err=d["flux_err"])

def phase_fold(t, period, t0):
    """Time -> phase in days, transit at 0, range [-period/2, period/2)."""
    return (t - t0 + period / 2) % period - period / 2

def bin_median(phase, flux, n_bins, lo, hi):
    med, edges, _ = binned_statistic(phase, flux, statistic="median", bins=n_bins, range=(lo, hi))
    return 0.5 * (edges[1:] + edges[:-1]), med

def flatten(lc, period, t0, dur_days):
    """Remove slow trends but ignore the transits while fitting, so the dips survive."""
    mask = lc.create_transit_mask(period=period, transit_time=Time(t0, format="bkjd", scale="tdb"), duration=1.5 * dur_days)
    win = int(max(3 * dur_days, 0.75) / (29.4 / 60 / 24)) // 2 * 2 + 1
    return lc.flatten(window_length=max(win, 101), mask=mask), mask

have = sub[sub.have_lc]
row = have[(have.av_training_set == "PC") & have.tce_period.between(2, 20)].sort_values("tce_model_snr", ascending=False).iloc[5]
P, t0, dur = row.tce_period, row.tce_time0bk, row.tce_duration / 24   # hours -> days
print(f"KIC {row.kepid}  ({row.av_training_set})  period {P:.3f} d, duration {dur*24:.1f} h, depth {row.tce_depth:.0f} ppm, SNR {row.tce_model_snr:.0f}")
lc = load_lc(row.kepid)

### 4a · The raw light curve

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(lc.time.value, lc.flux.value, ".", ms=1, color="k", alpha=0.4)
ax.set(xlabel="time [BKJD days]", ylabel="normalised flux", title=f"KIC {row.kepid}: four years of Kepler data")
plt.tight_layout()

**What you should see:** a band of points around 1.0, about 1,460 days long, with **gaps** between Kepler's ~90-day quarters (the spacecraft turned every quarter to point its solar panels at the Sun).

You may see slow wiggles or steps. That's the star's own variability (spots rotating in and out of view) and leftover instrument drift. You probably **can't see the transits at all** at this zoom. They're tiny and last only hours.

### 4b · Zoom in on a few transits

In [ ]:
n0 = np.ceil((lc.time.value.min() + 50 - t0) / P)
t_first = t0 + n0 * P
win = (lc.time.value > t_first - 1.5 * P) & (lc.time.value < t_first + 3.5 * P)
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(lc.time.value[win], lc.flux.value[win], ".", ms=2, color="k")
for n in range(-1, 4):
    ax.axvline(t_first + n * P, color="tab:green", alpha=0.5, ls="--")
ax.set(xlabel="time [BKJD days]", ylabel="normalised flux", title="A few orbits, with the predicted transit times (green dashed)")
plt.tight_layout()

**What you should see:** small dips lined up with the green lines, once per orbit. For a high-SNR planet they're clearly visible. For most TCEs they're buried in the noise. That's why the pipeline folds hundreds of transits on top of each other.

### 4c · Detrending (flattening)

In [ ]:
flat, mask = flatten(lc, P, t0, dur)
trend = lc.flux.value / flat.flux.value

fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
ax[0].plot(lc.time.value, lc.flux.value, ".", ms=1, color="k", alpha=0.3, label="raw")
ax[0].plot(lc.time.value, trend, color="tab:orange", lw=1, label="fitted trend")
ax[0].legend(loc="upper right"); ax[0].set_ylabel("raw flux")
ax[1].plot(flat.time.value, flat.flux.value, ".", ms=1, color="k", alpha=0.3)
ax[1].plot(flat.time.value[mask], flat.flux.value[mask], ".", ms=1.5, color="tab:green", label="in transit (masked from the fit)")
ax[1].legend(loc="upper right"); ax[1].set(ylabel="flattened flux", xlabel="time [BKJD days]")
plt.tight_layout()

**What you should see:** in the top plot, the orange trend follows the slow wiggles but **not** the transits. In the bottom plot, a flat band at 1.0 with the in-transit points (green) hanging slightly below it.

If the trend dipped into the transits, the flattening would erase them. That's the most common preprocessing bug, and the reason the transits are masked before the fit.

### 4d · Folding: the global and local views

In [ ]:
phase = phase_fold(flat.time.value, P, t0)
f = flat.flux.value

gx, gy = bin_median(phase, f, 2001, -P / 2, P / 2)
lx, ly = bin_median(phase, f, 201, -2 * dur, 2 * dur)

fig, ax = plt.subplots(1, 2, figsize=(15, 4.2))
ax[0].plot(phase, f, ".", ms=0.5, color="k", alpha=0.15)
ax[0].plot(gx, gy, color="tab:blue", lw=1.2)
ax[0].set(title="Global view: the whole orbit (2,001 bins)", xlabel="phase [days from transit]", ylabel="flux")
ax[1].plot(phase[np.abs(phase) < 2 * dur], f[np.abs(phase) < 2 * dur], ".", ms=1.5, color="k", alpha=0.25)
ax[1].plot(lx, ly, color="tab:blue", lw=2)
ax[1].set(title="Local view: zoomed on the transit (201 bins)", xlabel="phase [days from transit]")
lo = np.nanmin(ly); ax[0].set_ylim(lo - 0.3 * (1 - lo), 1 + 0.6 * (1 - lo))
plt.tight_layout()

**What you should see:** this is the key picture of the whole project.

- **Global view (left):** a flat line with one narrow dip at phase 0. Every transit from four years now sits on top of the others. A dip that was invisible in a single orbit is now obvious.
- **Local view (right):** the same dip, magnified. A planet gives a **U shape**: a quick drop as the planet enters the star's disc (*ingress*), a fairly flat bottom while it's fully in front, and a quick rise (*egress*). The bottom curves a little because stars are darker at the edges (*limb darkening*).

The blue lines, the medians per bin, are exactly the two arrays the CNN will get as input.

## 5 · Planets vs. false positives, side by side

The function below draws the global and local view for any TCE, plus an **odd/even** comparison (explained in the next section). Run it for several examples of each class.

In [ ]:
def show_tce(row, ax_row=None):
    P, t0, dur = row.tce_period, row.tce_time0bk, row.tce_duration / 24
    lc = load_lc(row.kepid)
    flat, _ = flatten(lc, P, t0, dur)
    t, f = flat.time.value, flat.flux.value
    phase = phase_fold(t, P, t0)
    n = np.round((t - t0) / P).astype(int)
    gx, gy = bin_median(phase, f, 2001, -P / 2, P / 2)
    lx, ly = bin_median(phase, f, 201, -2 * dur, 2 * dur)
    ox, oy = bin_median(phase[n % 2 == 1], f[n % 2 == 1], 61, -1.5 * dur, 1.5 * dur)
    ex, ey = bin_median(phase[n % 2 == 0], f[n % 2 == 0], 61, -1.5 * dur, 1.5 * dur)

    axs = ax_row if ax_row is not None else plt.subplots(1, 3, figsize=(16, 3.4))[1]
    c = COLORS[row.av_training_set]
    axs[0].plot(gx / P, gy, color=c, lw=1)
    axs[0].axvspan(0.45, 0.5, color="orange", alpha=0.15); axs[0].axvspan(-0.5, -0.45, color="orange", alpha=0.15)
    axs[0].set_title(f"KIC {row.kepid} · {row.av_training_set} · P={P:.2f} d · {row.tce_depth:.0f} ppm", fontsize=10)
    axs[0].set_xlabel("phase [orbits]  (orange = half an orbit later)")
    axs[1].plot(lx * 24, ly, color=c, lw=1.8); axs[1].set_xlabel("hours from transit"); axs[1].set_title("local view", fontsize=10)
    axs[2].plot(ox * 24, oy, color="tab:purple", lw=1.5, label="odd transits")
    axs[2].plot(ex * 24, ey, color="tab:olive", lw=1.5, label="even transits")
    axs[2].legend(fontsize=8); axs[2].set_xlabel("hours from transit"); axs[2].set_title("odd vs even", fontsize=10)

def gallery(label, n=4, seed=0):
    rows = have[have.av_training_set == label].sample(n, random_state=seed)
    fig, axes = plt.subplots(n, 3, figsize=(16, 2.9 * n))
    for r, ax_row in zip(rows.itertuples(), axes):
        try:
            show_tce(r, ax_row)
        except Exception as e:
            ax_row[0].set_title(f"KIC {r.kepid}: {e}")
    plt.tight_layout()

### 5a · Planet candidates (PC)

In [ ]:
gallery("PC", n=4, seed=1)

**What you should see:** U-shaped dips, mostly shallow; **nothing** in the orange band half an orbit later; odd and even transits lying **on top of each other**.

Some PCs look almost flat in the local view. Those are low-SNR planets where even four years of folding barely shows the dip. They are the hard cases for the model too.

### 5b · Astrophysical false positives (AFP), mostly eclipsing binaries

In [ ]:
gallery("AFP", n=4, seed=1)

**What you should see:** at least some of these give the game away:

- **Deep, V-shaped dips:** two stars grazing each other rather than a small planet fully crossing.
- **A second, smaller dip in the orange band, half an orbit later (a *secondary eclipse*):** the companion is itself a star, so you also see it being eclipsed. A planet is too dark to give a visible secondary.
- **Odd and even depths that differ:** Kepler found the binary at *half* its real period, so it folds two different stars' eclipses together. Alternating transits then have different depths.

Some AFPs look exactly like planets. These are usually a binary in the background, blended with the target star, which makes its deep eclipse look shallow. Even humans needed extra data (like a shift in the star's position during transit) to catch those.

### 5c · Non-transiting phenomena (NTP)

In [ ]:
gallery("NTP", n=4, seed=1)

**What you should see:** no clean, repeating dip. Typical patterns are noise with a slight bump at phase 0; a sinusoidal wave across the whole orbit (a spotted star rotating, or a pulsating star); or a single sharp feature, from an instrument glitch that happened to line up.

These are usually the **easiest** class for the model. The hard boundary is PC vs AFP.